## Cell 0. API keys

Paste your keys between the quotes below and run this cell before anything
else. Leave a line as `""` to use whatever is already exported in the
environment instead.

**Two things worth knowing before you paste.** This notebook is regenerated by
`build_q1_nb.py`, which rewrites every cell from source -- so a key typed here
is lost on the next rebuild. And a key typed here is saved inside the `.ipynb`
file, where it can reach git or a shared copy. For a key you intend to keep,
put it in `fourarm/env/keys.local.env` instead, which this cell reads
automatically and which is gitignored and never regenerated.

In [1]:
# --- Cell 0. API keys. Run first. -------------------------------------------
import os, pathlib

# PASTE BETWEEN THE QUOTES. Leave "" to fall back to the environment or to
# env/keys.local.env.
KEYS = {
    "OPENAI_API_KEY": "",
    "GEMINI_API_KEY": "",
    "ANTHROPIC_API_KEY": "",
}

# An EMPTY value must never be written into the environment. Assigning ""
# unconditionally would blank a key that is already exported correctly, and
# the failure -- a 401 from a variable that is set but empty -- reads nothing
# like "you left the placeholder alone".
for _name, _value in KEYS.items():
    if _value.strip():
        os.environ[_name] = _value.strip()

# The persistent alternative. Same KEY=value format as env/models.env, one
# per line, # for comments. Read only for names not already set, so anything
# pasted above and anything already exported both win over the file.
_here = pathlib.Path.cwd()
_root = next((c for c in [_here] + list(_here.parents)
              if (c / "out").is_dir() and (c / "experiments").is_dir()), None)
_local = _root / "env" / "keys.local.env" if _root else None
if _local and _local.exists():
    for _line in _local.read_text().splitlines():
        _line = _line.strip()
        if not _line or _line.startswith("#") or "=" not in _line:
            continue
        _k, _, _v = _line.partition("=")
        _k, _v = _k.strip(), _v.strip().strip("\'\"")
        if _v and not os.environ.get(_k):
            os.environ[_k] = _v

# Report presence, NEVER the value. Printing a key would write it into the
# notebook's saved output, which is the same leak as pasting it into a cell
# and is easier to do by accident.
#
# The report loops over KEYS, so EVERY key the notebook can use needs a row
# there even when it is only ever supplied by keys.local.env. A name missing
# from KEYS still loads from the file, but silently, and a key that loads
# without being reported is indistinguishable from one that did not load.
for _name in KEYS:
    _set = bool(os.environ.get(_name))
    print("%-18s %s" % (_name, "set" if _set else "NOT SET"))
if _local:
    print("%-18s %s" % ("keys.local.env",
                        "read" if _local.exists() else "absent (optional)"))

OPENAI_API_KEY     set
GEMINI_API_KEY     set
ANTHROPIC_API_KEY  set
keys.local.env     read


# Experiment 2, Q3 Repair

Two steps, each addressed twice: once by supplying information the model may be
missing, once by rephrasing the question while supplying nothing. Two of the
four cells are already on disk from the ladder; this notebook buys the other
two, plus two controls.

Run order is controls, read-out, then treatments. The read-out refuses to
report a treatment until both controls have landed and passed.

Cells 0 to 3 are Q1's, unchanged, so the sample and the exclusions are the
same ones every other Experiment 2 notebook uses.

In [2]:
# --- Cell 1. Setup. No model calls. -----------------------------------------
import collections, csv, datetime, hashlib, json, math, os, pathlib, sys

# Find the package root: the directory holding out/ and experiments/.
here = pathlib.Path.cwd()
ROOT = None
for cand in [here] + list(here.parents):
    if (cand / "out").is_dir() and (cand / "experiments").is_dir():
        ROOT = cand
        break
if ROOT is None:
    raise SystemExit("run this from fourarm/ or below: no out/ + experiments/ found")
for p in (str(ROOT), str(ROOT / "ycb")):
    if p not in sys.path:
        sys.path.insert(0, p)

# RE-IMPORT, never reuse. Python caches modules in sys.modules, so running
# this cell a second time in a live kernel keeps whatever was on disk the
# FIRST time it ran. While the ex2 modules are being edited alongside the
# notebook that is a trap: the kernel holds the old vocabulary, and the
# failure surfaces cells later as a design-check assertion naming a face
# that no longer exists, which reads like a code error and is not one.
#
# Dropping the entries and importing fresh is used rather than
# importlib.reload because these modules import each other, and reload
# leaves a half-updated graph unless the order is exactly right.
for _stale in [m for m in list(sys.modules)
               if m.startswith(("experiments.ex2", "analysis.ex2"))
               or m in ("ycb_objects",)]:
    del sys.modules[_stale]

# WHICH KERNEL THIS IS, checked before the first project import.
#
# The very next line reaches core.decision.state_builder through
# mancheck -> vlm_allocator, and that imports numpy; visibility.py, in cell
# 3, needs PIL and scipy. On a kernel without them the notebook dies forty
# lines deep inside somebody else's module with "No module named 'numpy'",
# which reads as a broken repository rather than as a kernel picked from a
# list of six. Checked here, where the answer is one sentence.
_missing = []
for _m in ("numpy", "PIL", "scipy"):
    try:
        __import__(_m)
    except ImportError:
        _missing.append(_m)
if _missing:
    _venv = ROOT.parent / ".venv" / "bin" / "python"
    raise SystemExit(
        "WRONG KERNEL.\n"
        "  This kernel is  %s\n"
        "  and it has no %s.\n"
        "  Use instead     %s\n"
        "  In VS Code: Select Kernel, then Python Environments, then the\n"
        "  interpreter at that path. It is the only one in this tree with\n"
        "  ipykernel AND numpy, PIL and scipy. Several unrelated kernels are\n"
        "  registered on this machine and any of them will get this far and\n"
        "  then fail."
        % (sys.executable, ", ".join(_missing), _venv))

from core.cell import cell_config as C
from core.decision import model_registry as MR
from experiments.ex2 import grade as G
from experiments.ex2 import labels as L
from experiments.ex2 import mancheck as MC
from experiments.ex2 import prompts as P
from experiments.ex2 import run as R
from experiments.ex2 import solo as S
from experiments.ex2 import transforms as T
from experiments.ex2 import visibility as VIS
from analysis.ex2.ex2_stats import newcombe, paired_mean_ci, spans_zero, wilson
# The notebook machinery: loaders, the share definition, the paired
# contrast and the spend gate. In a module rather than in this cell so
# that Q2 and Q3 use the same ones rather than a second copy, and so
# that harness/h_ex2_q_common.py can pin them. What stays in the cells
# is what is a DECISION: the models, the rung, the conditions, the
# usable rule, each cost, and every CONFIRM_SPEND.
from analysis.ex2.ex2_q_common import (Outputs, answered,       # noqa
                                       coupling, fmt, full_flip_count,
                                       is_franka, keep_analysable,
                                       load_run, paired_delta,
                                       paired_diffs, pct,
                                       provenance_row, run_meta,
                                       sha256, share_at, share_counts,
                                       show, spend_gate)

# --- paths ------------------------------------------------------------------
CAPTURES = ROOT / "out" / "ex2_capture_block"
RUNS     = ROOT / "runs"
TABLES   = ROOT / "tables" / "ex2_q3_repair"
FIGURES  = ROOT / "figures" / "ex2_q3_repair"
for d in (RUNS, TABLES, FIGURES):
    d.mkdir(parents=True, exist_ok=True)

# --- constants, every one read from a source of truth ------------------------
RUNG        = None                       # each paid cell names its own
PREFERENCE  = "franka"
CONDITIONS  = ("congruent_face", "dims")
REPEATS     = 3
FACES       = P.RESTING_FACES            # small_face, large_face
LABEL       = "ycb_block"

FRANKA_MAX  = C.ARM_TYPES["franka"]["max_grasp_m"]
UR_MAX      = C.ARM_TYPES["ur10"]["max_grasp_m"]
DIMS        = T.DIMS_M[LABEL]
FACTS       = T.POSE_FACTS_BY_LABEL[LABEL]

# The registry has no hardcoded model list: aliases() reads FOURARM_MODELS.
try:
    ALIASES = MR.aliases()
except Exception as exc:
    ALIASES = []
    print("model registry unavailable (%s); set MODELS by hand below" % exc)
# THREE models since 2026-08-27. claude-sonnet-5 was added because the
# design needs a third model that CLEARS the two-way face probe: with two
# models, a single failure at cell 5b leaves one, and one model cannot show
# that a result is a property of models rather than of this one model.
# It is not here for being the most capable available; see env/models.env.
#
# claude_md RATHER THAN claude. Same model, claude-sonnet-5, at effort
# medium instead of the API default of high. At the default it read the
# two-way face probe at 65 percent against gpt's 95 and gemini's 100, and
# it failed by BIAS rather than blindness: large_face on 75 percent of
# trials, 90 percent right when the block lies flat and 40 percent when it
# stands. Deliberation is how a prior like "blocks lie flat" gains weight,
# so lower effort is the move that fits the failure. Cell 5b is what tests
# it. The default-effort runs stay on disk under the alias "claude".
#
# gpt_hi RATHER THAN gpt. Same model, gpt-5.6-terra, at reasoning_effort
# high instead of low. Claude runs at the Anthropic default effort of high
# and Gemini Flash exposes no effort control at all, so gpt at low made the
# one model with the LEAST test-time compute the yardstick for the other
# two. Effort is still not matched across providers and cannot be -- that
# stays in Limitations -- but the reasoning models are now on the same
# nominal tier.
#
# The low-effort runs are NOT deleted. runs/ex2_q1_cue2way_gpt_r*.jsonl
# record gpt at reasoning_effort low over this same sample and stay on disk
# as the evidence for what effort was worth here: 95 percent at low. A
# separate alias rather than an edit to GPT_PARAMS is what makes those rows
# still readable, which is the reason env/models.env gives for gpt_hi
# existing at all.
#
# Named rather than taken wholesale from ALIASES. FOURARM_MODELS also lists
# qwen and gpt_hi, and a run's model set must be a decision recorded here,
# not whatever the registry happens to carry. The fallback keeps the same
# three so a registry failure cannot silently shrink the design.
_WANT = ("gpt_hi", "claude_md")
MODELS = tuple(a for a in ALIASES if a in _WANT) or _WANT
if set(MODELS) != set(_WANT):
    print("WARNING: %s requested, %s available from the registry. Every "
          "table below is per model, so a missing one narrows the design "
          "rather than breaking it -- but say so in the chapter."
          % (list(_WANT), list(MODELS)))

# --- credentials: reported, not assumed -------------------------------------
# Until 2026-08-27 a hand-added launcher cell started JupyterLab in a browser
# and refused to launch when a key was missing. That cell is gone: it spawned
# a NEW server every time it ran, which is how eight of them accumulated, each
# serving its own in-memory copy of this notebook, so an edit on disk could be
# invisible in the tab you were typing in. VS Code runs the kernel directly
# and needs no launcher -- but the key check it performed was worth keeping,
# so it lives here.
#
# This REPORTS rather than raises. Cells 1-5 and every analysis cell make no
# model calls and must stay runnable with no key at all. What it buys is
# learning about a missing key now instead of at cell 6, part-way into a run.
#
# The usual cause is launching VS Code from Finder or the Dock, which does not
# inherit a login shell, so a key exported in .zshrc is absent here while
# present in any terminal. The message below says so, because the symptom
# otherwise looks like a broken registry.
#
# key_var is read from the registry, never hardcoded: models.env lets each
# alias name its own variable, and a hardcoded OPENAI_API_KEY would check the
# wrong one the moment that is used.
MISSING_KEYS = []
for _alias in MODELS:
    try:
        _var = MR.describe(_alias)["key_var"]
    except Exception as _exc:
        MISSING_KEYS.append("%s: %s" % (_alias, _exc))
        continue
    if not os.environ.get(_var):
        MISSING_KEYS.append("%s: %s is not set" % (_alias, _var))

# Output paths travel together in one object, so a notebook cannot end up
# with a root and a tables directory that disagree. Rebound to bare names
# because every call site below reads better as write_csv(...) than as
# OUT.write_csv(...), and because leaving those call sites untouched is
# what made this extraction verifiable against the tables already on disk.
OUT = Outputs(ROOT, TABLES, FIGURES)
rel, write_csv = OUT.rel, OUT.write_csv

print("root        ", ROOT)
print("captures    ", CAPTURES.relative_to(ROOT), "(exists:", CAPTURES.is_dir(), ")")
print("rung        ", RUNG, " preference", PREFERENCE, " repeats", REPEATS)
print("models      ", MODELS, " (registry knows: %s)" % (ALIASES or "nothing"))
print("prompt ver  ", P.EX2_PROMPT_VERSION)
# Printed, not assumed. If a stale kernel ever slips past the re-import
# above, this is the line that shows it, at the top of the run rather than
# in an assertion twenty cells later.
print("faces       ", FACES, " chance %.1f%%" % (100.0 / len(FACES)))
if MISSING_KEYS:
    print("api keys     MISSING -- analysis runs, model calls will not:")
    for _m in MISSING_KEYS:
        print("               ", _m)
    print("             launch VS Code from a shell that exports them:")
    print("               open -a 'Visual Studio Code' <repo>")
else:
    print("api keys     present for %s" % (", ".join(MODELS),))
print()
print("block           %.3f x %.3f x %.3f m"
      % (DIMS["height"], DIMS["width"], DIMS["depth"]))
print("franka opens to  %.3f m   ur opens to %.3f m" % (FRANKA_MAX, UR_MAX))
print("resting faces   %s" % (FACES,))
for f in FACES:
    print("   %-11s needs %.3f m" % (f, FACTS[f]["grasp_m"]))

root         /Users/erinsarlak/Downloads/MastersDissertation/fourarm
captures     out/ex2_capture_block (exists: True )
rung         None  preference franka  repeats 3
models       ('gpt_hi', 'claude_md')  (registry knows: ['qwen', 'gpt', 'gpt_hi', 'gemini', 'claude', 'claude_md'])
prompt ver   2026-08-27b
faces        ('small_face', 'large_face')  chance 50.0%
api keys     present for gpt_hi, claude_md

block           0.130 x 0.100 x 0.050 m
franka opens to  0.080 m   ur opens to 0.140 m
resting faces   ('small_face', 'large_face')
   small_face  needs 0.050 m
   large_face  needs 0.100 m


## Cell 2. Design check

No model calls. Derives the opening for each resting face from the authored
cuboid dimensions and asserts it matches what `transforms` declares. The prompt
states the bounding-box convention, so a divergence here would make the prompt
wrong rather than silent.

In [3]:
# --- Cell 2. Design check. No model calls. ----------------------------------
from ycb_objects import YCB as _SPECS      # the authored object dictionary

# STALE-IMPORT GUARD. Cell 1 purges sys.modules before importing, so a
# module edited on disk is picked up whenever cell 1 is re-run. This
# catches the case where cell 1 was NOT re-run -- editing a module and
# jumping straight back to this cell -- and the worse case where the
# NOTEBOOK ITSELF is stale, because JupyterLab holds its own copy in the
# browser and does not re-read the file when it changes underneath. A
# stale cell 1 has no purge, so the modules stay old and the design
# assertion below fails naming a face that no longer exists. That reads
# like a code error and is not one, which is why this checks first and
# says which of the two it is.
#
# The comparison is against the SOURCE ON DISK, not against a constant
# written here, so it stays true across future vocabulary changes.
import re                                   # local: a stale Cell 1 may
                                            # not have imported it
_src = pathlib.Path(P.__file__).read_text()
_on_disk = re.search(r'EX2_PROMPT_VERSION\s*=\s*["\'](.+?)["\']', _src)
if _on_disk and _on_disk.group(1) != P.EX2_PROMPT_VERSION:
    raise SystemExit(
        "STALE IMPORT: this kernel holds prompts.py version %s, but the file "
        "on disk is %s.\n"
        "  Loaded faces: %s\n"
        "  Fix: re-run Cell 1, which drops the cached modules and imports "
        "fresh.\n"
        "  If re-running Cell 1 does not clear it, the NOTEBOOK is stale, not "
        "the kernel:\n"
        "  JupyterLab is running the copy it loaded into the browser. Use "
        "File > Reload\n"
        "  Notebook from Disk, then Restart Kernel and Run All."
        % (P.EX2_PROMPT_VERSION, _on_disk.group(1), list(FACES)))

# Each block prim is spawned already resting on a face, with size PRE-ORIENTED
# to that pose: size is (x, y, z) with z vertical. So the two horizontal
# extents are size[0] and size[1], and the opening is the smaller of them.
# YCB is keyed without the "ycb_" scene prefix.
PRIM_OF_FACE = {L.TRUE_POSE[p]: p for p, lab in L.POSE_ENTRIES.items()
                if lab == LABEL}

design_rows = []
problems = []
for face in FACES:
    prim = PRIM_OF_FACE[face]
    size = _SPECS[prim.replace("ycb_", "")]["size"]
    horiz = sorted(size[:2], reverse=True)          # a = larger, b = smaller
    vertical = size[2]
    opening = min(horiz)
    declared = FACTS[face]["grasp_m"]

    if abs(opening - declared) > 1e-9:
        problems.append("%s: bounding box gives %.3f, transforms declares %.3f"
                        % (face, opening, declared))
    # A real permutation check, all three extents. It compared only the
    # SMALLEST until 2026-08-27, so a prim sized 0.200 x 0.200 x 0.050 --
    # not the block at all -- passed a check whose message said it was
    # verifying a permutation. That matters more with two faces than it
    # did with three: there are fewer cross-checks left, and this cell is
    # what stands between a mis-authored prim and the whole experiment.
    if (sorted(round(v, 6) for v in list(horiz) + [vertical])
            != sorted(round(DIMS[k], 6) for k in ("height", "width", "depth"))):
        problems.append(
            "%s: extents %s are not a permutation of the block %s"
            % (face, sorted(list(horiz) + [vertical]),
               sorted(DIMS[k] for k in ("height", "width", "depth"))))

    franka_ok = declared <= FRANKA_MAX
    ur_ok = declared <= UR_MAX
    design_rows.append([face, "%.3f" % vertical, "%.3f" % horiz[0],
                        "%.3f" % horiz[1], "%.3f" % declared,
                        "%.3f" % FRANKA_MAX, "%.3f" % UR_MAX,
                        franka_ok, ur_ok,
                        "franka, preference satisfied" if franka_ok
                        else "UR, preference overridden"])

# The design only works if the Franka is feasible on one face and not the
# other, and the UR on both. Anything else and Q1 has no contrast.
feasible = [r[0] for r in design_rows if r[7]]
if sorted(feasible) != ["small_face"]:
    problems.append("franka feasible on %s, expected small_face alone"
                    % sorted(feasible))
if not all(r[8] for r in design_rows):
    problems.append("a UR is infeasible somewhere; it must be legal everywhere")

show(["face", "vert", "horiz_a", "horiz_b", "opening", "franka", "ur", "picks"],
     [[r[0], r[1], r[2], r[3], r[4], r[7], r[8], r[9]] for r in design_rows])
print()
if problems:
    raise AssertionError("DESIGN CHECK FAILED:\n  " + "\n  ".join(problems))
print("PASS  every opening is the smaller horizontal extent, and the Franka")
print("      is feasible on small_face and not on large_face.")
print("      A third face, the middle one, was withdrawn on 2026-08-27: it")
print("      was flat like large_face and differed only in geometry, which")
print("      made it the sharper test, but no model read it (GPT 58%,")
print("      Fisher p=0.76 over 81 trials). The cost is that a model")
print("      reading posture and applying a rule can no longer be told")
print("      apart from one deriving the opening from geometry.")

write_csv("tab_ex2_q1_design.csv",
          ["resting_face", "vertical_m", "horiz_a_m", "horiz_b_m",
           "opening_needed_m", "franka_max_m", "ur_max_m", "franka_feasible",
           "ur_feasible", "deriving_model_picks"],
          design_rows)

face        vert   horiz_a  horiz_b  opening  franka  ur    picks                       
----------  -----  -------  -------  -------  ------  ----  ----------------------------
small_face  0.130  0.100    0.050    0.050    True    True  franka, preference satisfied
large_face  0.050  0.130    0.100    0.100    False   True  UR, preference overridden   

PASS  every opening is the smaller horizontal extent, and the Franka
      is feasible on small_face and not on large_face.
      A third face, the middle one, was withdrawn on 2026-08-27: it
      was flat like large_face and differed only in geometry, which
      made it the sharper test, but no model read it (GPT 58%,
      Fisher p=0.76 over 81 trials). The cost is that a model
      reading posture and applying a rule can no longer be told
      apart from one deriving the opening from geometry.
wrote tables/ex2_q3_repair/tab_ex2_q1_design.csv  (2 rows)


PosixPath('/Users/erinsarlak/Downloads/MastersDissertation/fourarm/tables/ex2_q3_repair/tab_ex2_q1_design.csv')

## Cell 3. Capture inventory and legality

No model calls. Loads the captures, checks the grid is complete, re-asserts the
recorded settle heights, and runs the **real validator** over every scene to
establish which positions can carry the contrast at all.

This cell is the reason the sample is 29 positions rather than 30, and it fails
loudly rather than letting the analysis assume otherwise.

**Why the settle heights are re-checked here.** The prompt never says which flat
orientation to expect. The convention sentence -- *an object resting flat lies on
its largest face* -- was deliberately left out, because capture enforces it
instead: `capture_ex2_scene.py` fails a capture that settles at the wrong height
rather than relabelling it with the face it was asked for. That assertion is real
and it does raise, but it post-dates most of the captures on disk, and the trail
check below it compares only the recorded face *word* against the prim -- never
the height that word is supposed to describe. So the single guarantee standing
behind the prompt's silence was being taken on trust at the point where the data
is actually read. Every capture records `ex2.settled[name]`, so checking it costs
nothing. The tolerance is read out of the capture script's source rather than
typed here, so it cannot drift from the value the captures were accepted under.

In [4]:
# --- Cell 3. Capture inventory and legality. No model calls. ----------------
import re                                   # local, as in Cell 2: Cell 1
                                            # does not import it, so this
                                            # cell must not depend on Cell 2
                                            # having been run first
scenes = R.load_scenes(str(CAPTURES))          # normalises the idle UR
raw    = R.load_scenes(str(CAPTURES), present_ur=False)   # as written
trail  = [json.loads(l) for l in open(CAPTURES / "consults.jsonl") if l.strip()]

by_pos = collections.defaultdict(dict)
for s in scenes:
    pos, member = s["seq"].rsplit("_", 1)
    prim = [o["name"] for o in s["state"]["objects"] if LABEL.split("_")[-1] in o["name"]][0]
    by_pos[pos][L.TRUE_POSE[prim]] = s

# DERIVED, never literal. This read "90 captures / 30 positions" until
# 2026-08-27 and raised the moment four positions were added to the capture
# plan. A count typed here goes stale silently; one derived from the
# directory cannot. What actually matters is not the total but that every
# position carries every face, which `missing` below checks.
inv_problems = []
if len(scenes) != len(by_pos) * len(FACES):
    inv_problems.append("expected %d captures (%d positions x %d faces), "
                        "found %d" % (len(by_pos) * len(FACES), len(by_pos),
                                      len(FACES), len(scenes)))
missing = {p: sorted(set(FACES) - set(v)) for p, v in by_pos.items()
           if set(v) != set(FACES)}
if missing:
    inv_problems.append("positions missing a face: %s" % missing)
if len({s["seq"] for s in scenes}) != len(scenes):
    inv_problems.append("duplicate seq ids")

# The face is derived from the PRIM, never from the trail's word, and the
# trail is then checked against it.
#
# Captures written before 2026-08-27 record ex2.resting_face in a superseded
# vocabulary where "upright" meant small_face and "small_face" meant the
# retired middle face. capture_ex2_scene.py now writes the geometric name
# directly, so new captures need no translation; this map reads the old ones
# and is why the check is against the prim rather than the word.
TRAIL_FACE = {"upright": "small_face", "small_face": "edge",
              "large_face": "large_face"}

# READ FROM THE CAPTURE SCRIPT, not typed here. capture_ex2_scene.py
# imports isaaclab at module scope and cannot be imported into this kernel,
# and a literal copied into the notebook would go stale the moment the
# tolerance is retuned -- which it was, from 0.010 to 0.005, on 2026-08-27.
# Same regex-the-source trick cell 2 uses for EX2_PROMPT_VERSION.
_cap_src = (ROOT / "ycb" / "capture_ex2_scene.py").read_text()
_tol = re.search(r"^SETTLE_TOL_M\s*=\s*([0-9.]+)", _cap_src, re.M)
if not _tol:
    raise SystemExit(
        "SETTLE_TOL_M not found in ycb/capture_ex2_scene.py. It is the "
        "tolerance the captures were accepted under and the notebook must "
        "not invent one; if it was renamed, update this cell.")
SETTLE_TOL_M = float(_tol.group(1))
for rec in trail:
    prim = [o["name"] for o in rec["state"]["objects"] if "block" in o["name"]][0]
    want = L.TRUE_POSE.get(prim)
    if want is None:
        continue          # a retired-face capture; load_scenes drops it too
    word = (rec.get("ex2") or {}).get("resting_face")
    # BOTH vocabularies are accepted, and only because each is checked
    # against the prim. A word is fine if it already IS the derived face
    # (written 2026-08-27 or later) or if it translates to it (written
    # before). Anything else is a genuine disagreement. Accepting both is
    # not laxity: the prim is the truth in either case, and the word is
    # never the thing consulted downstream.
    if word != want and TRAIL_FACE.get(word) != want:
        inv_problems.append("%s: trail says %r, prim says %r"
                            % (rec["seq"], word, want))

    # THE SETTLE HEIGHTS, RE-ASSERTED WHERE THE DATA IS READ.
    #
    # The prompt says nothing about which flat orientation to expect. The
    # convention sentence ("an object resting flat lies on its largest
    # face") was deliberately NOT added, on the grounds that capture
    # enforces it instead -- and it does: capture_ex2_scene.py raises on a
    # capture that settles at the wrong height rather than labelling it
    # with the face it was asked for. But that assertion post-dates most
    # captures on disk, and the check above compares only the trail's face
    # WORD against the prim, never the height that word is supposed to
    # describe. So the one guarantee standing behind the prompt's silence
    # was, at this point, taken on trust. It is not expensive to check.
    for name, d in (rec.get("ex2") or {}).get("settled", {}).items():
        z, want_z = d.get("z_above_table"), d.get("expected_rest_z")
        if want_z is None:
            inv_problems.append("%s: %s has no expected_rest_z, so its "
                                "resting face was never verified"
                                % (rec["seq"], name))
        elif abs(z - want_z) > SETTLE_TOL_M:
            inv_problems.append(
                "%s: %s settled at z=%.4f, expected %.4f within %.3f. It is "
                "not on the face this capture claims."
                % (rec["seq"], name, z, want_z, SETTLE_TOL_M))

print("captures %d   positions %d   faces per position %s"
      % (len(scenes), len(by_pos),
         sorted({len(v) for v in by_pos.values()})))
print("idle UR presented: %s"
      % dict(collections.Counter(s["idle_ur"] for s in scenes)))
print("idle UR as captured: %s"
      % dict(collections.Counter(
          tuple(sorted(a["name"] for a in s["state"]["arms"]
                       if a["state"] == "IDLE" and a["name"].startswith("ur")))
          for s in raw)))
print()

# --- the real validator, per scene ------------------------------------------
legal = {}
for pos in sorted(by_pos):
    for face, s in by_pos[pos].items():
        st, meta = T.transform({"state": s["state"],
                                "positions_exact": s["positions_exact"]},
                               "congruent")
        tid = R.flip_task_id(s["state"], meta["flip_prim"])
        legal[(pos, face)] = sorted(R.legal_arms(s, meta["flip_prim"], tid))

def has_franka(arms):
    return any(a.startswith("franka") for a in arms)

# TWO independent preconditions, not one. Legality asks whether the aperture
# contrast EXISTS at a position; visibility asks whether the block can be
# SEEN there. A position can carry the full contrast with the block hidden
# behind the Franka, and until 2026-08-27 nothing noticed: e10 presents 3%
# of the median block area and GPT inverted both its posture trials.
#
# visibility.verdict reads pixels only and never a model reply, so a
# position is never excluded for having scored badly.
#
# Run PER PREFIX, not over the directory at once. Each pose is scored
# against the median of its own pose, and the west and east banks sit at
# different distances from the camera: a w block renders about 10 percent
# larger than an e block in the same pose. One pooled median would raise
# the bar for the far bank and lower it for the near one, which is a
# comparison between banks rather than a test of occlusion.
OCCLUDED, _vis_detail = [], {}
for _pfx in sorted({p[0] for p in by_pos}):
    _c = VIS.measure(str(CAPTURES), prefix=_pfx)
    _ok, _bad, _d = VIS.verdict(_c)
    print("visibility, %s bank:" % _pfx)
    print(VIS.report(_d, _ok, _bad))
    print()
    OCCLUDED += _bad
    _vis_detail.update(_d)

USABLE, excluded = [], {}
for pos in sorted(by_pos):
    sm, lg = (legal[(pos, f)] for f in ("small_face", "large_face"))
    ok = has_franka(sm) and lg and not has_franka(lg)
    if ok:
        USABLE.append(pos)
    else:
        excluded[pos] = {"small_face": sm, "large_face": lg}

show(["position", "small_face", "large_face", "usable"],
     [[pos, ",".join(legal[(pos, "small_face")]) or "NONE",
       ",".join(legal[(pos, "large_face")]) or "NONE",
       "yes" if pos in USABLE else "NO"] for pos in sorted(by_pos)])
print()
USABLE = [p for p in USABLE if p not in OCCLUDED]
print("positions carrying the full contrast and showing the block: %d of %d"
      % (len(USABLE), len(by_pos)))
for pos in sorted(set(OCCLUDED)):
    print("  EXCLUDED %s  the block is occluded here (%.2f of the pose"
          % (pos, _vis_detail[pos]["worst"]))
    print("           median, worst in %s). The contrast may exist, but a"
          % _vis_detail[pos]["worst_pose"])
    print("           perception result from a picture that does not show")
    print("           the object is not a result about the model.")
for pos, v in excluded.items():
    print("  EXCLUDED %s  %s" % (pos, v))
    print("           the franka is never legal here, so there is no arm choice")
    print("           to make and no contrast to measure. Excluded with cause,")
    print("           not dropped silently.")

write_csv("tab_ex2_q1_inventory.csv",
          ["position", "usable", "occluded", "idle_ur", "legal_small_face",
           "legal_large_face"],
          [[pos, pos in USABLE, pos in OCCLUDED,
            by_pos[pos]["small_face"]["idle_ur"],
            ";".join(legal[(pos, "small_face")]),
            ";".join(legal[(pos, "large_face")])] for pos in sorted(by_pos)])

if inv_problems:
    raise AssertionError("INVENTORY FAILED:\n  " + "\n  ".join(inv_problems))
if len(USABLE) < 20:
    raise AssertionError("only %d usable positions; the contrast is not "
                         "estimable and the run should not be paid for"
                         % len(USABLE))

# --- what the PAID cells ask about ------------------------------------------
# Defined here, ONCE, and used by both cell 6 and cell 7. Putting the choice
# in each paid cell would let the two be set differently, so congruent and
# dims would cover different scene sets and the paired contrast in cell 10
# would silently compare two different samples.
#
# TRUE is the standing decision: ask about every captured scene, including
# the excluded positions, and drop them in cell 8 at ANALYSIS time. It costs
# a little more and buys something the write-up needs -- the excluded rows
# are in the data, so the exclusion can be shown to predate any accuracy
# result rather than being read as a position dropped for scoring badly.
#
# Set FALSE to pay only for the usable positions. The analysis is unaffected
# either way: cell 8 restricts to USABLE regardless.
RUN_ALL_POSITIONS = True

CALL_SCENES = ([s for s in scenes
                if s["seq"].rsplit("_", 1)[0] in USABLE]
               if not RUN_ALL_POSITIONS else scenes)

print()
print("PASS  inventory complete, %d positions usable." % len(USABLE))
print("      paid cells will ask about %d scenes (%s)"
      % (len(CALL_SCENES),
         "all captured, excluded positions included on purpose"
         if RUN_ALL_POSITIONS else "usable positions only"))

[ex2] 34 capture(s) skipped: they rest on a face this design no longer uses, and are kept on disk as evidence. e00_S, e01_S, e02_S, e03_S ...
[ex2] 34 capture(s) skipped: they rest on a face this design no longer uses, and are kept on disk as evidence. e00_S, e01_S, e02_S, e03_S ...
captures 68   positions 34   faces per position [2]
idle UR presented: {'ur_w': 34, 'ur_e': 34}
idle UR as captured: {('ur_w',): 68}



visibility, e bank:
pos          L        S        U      worst  verdict
e10        224       51       45       0.03  OCCLUDED (U)
e05        488      893     1064       0.60  ok
e04        719     1380     1258       0.83  ok
e01        650     1273     1325       0.85  ok
e16        655     1311     1302       0.86  ok
e13        661     1327     1356       0.89  ok
e06        685     1438     1390       0.92  ok
e15        720     1483     1549       0.99  ok
e12        736     1497     1509       0.99  ok
e02        717     1499     1518       1.00  ok
e11        727     1495     1530       1.00  ok
e14        774     1551     1567       1.03  ok
e07        758     1562     1600       1.04  ok
e03        798     1564     1644       1.05  ok
e00        811     1650     1625       1.07  ok
e09        828     1709     1703       1.12  ok
e08        840     1741     1738       1.14  ok

cut at 0.50 of the pose median, in every pose.
worst excluded 0.03, best retained 0.60: the cut sits

visibility, w bank:
pos          L        S        U      worst  verdict
w01        646     1266     1326       0.84  ok
w10        663     1326     1374       0.88  ok
w16        669     1347     1363       0.88  ok
w07        679     1360     1369       0.89  ok
w05        687     1432     1424       0.92  ok
w15        719     1491     1503       0.97  ok
w02        720     1486     1517       0.97  ok
w13        720     1494     1541       0.97  ok
w11        768     1505     1507       0.98  ok
w14        741     1564     1563       1.00  ok
w04        758     1622     1595       1.02  ok
w03        787     1559     1587       1.03  ok
w00        802     1650     1620       1.05  ok
w06        800     1602     1621       1.05  ok
w09        803     1623     1627       1.06  ok
w12        824     1705     1717       1.11  ok
w08        855     1774     1757       1.14  ok

cut at 0.50 of the pose median, in every pose.
17 usable, 0 occluded: none
this reads pixels only and never a 

## Controls. Run these first.

**Make model calls,** one repeat each, about \num{272} calls in total.

Neither control is a treatment. Each is a configuration already characterised
by earlier work, re-bought now so that the treatments have something to be
measured against.

**Control A**, `congruent_face` at `N-CD`, supplies the face and the rule and
withholds only the opening. GPT reached a contrast of 89.6 on this condition at
`N0`; with the rule added it cannot do worse for a reason that is about the
models. A low value here means the instrument, not the model.

**Control B**, `dims` at `N-CD`, is the configuration both step-1 treatments are
read against. It must reproduce the 32.3 already reported for GPT and the 0.0
for Claude. A departure means the models have changed since those runs and that
no treatment below can be compared with them.

Both resume: `solo.run` appends and flushes each row as it lands and skips
every trial already answered, so interrupting a cell costs nothing.

In [5]:
# --- Control A. congruent_face at N-CD. MAKES MODEL CALLS. ------------------
CTRL_A_OUT = RUNS / "ex2_q3_ctrlA_congruent_face_N-CD.jsonl"
CTRL_REPEATS = 1          # bound locally: a later cell rebinds REPEATS

n_calls = len(CALL_SCENES) * len(MODELS) * CTRL_REPEATS
print("COST: %d scenes x %d models x %d repeat = %d calls"
      % (len(CALL_SCENES), len(MODELS), CTRL_REPEATS, n_calls))
print("      congruent_face at N-CD. The face and the rule are both given,")
print("      so only the conversion is at stake. GPT read 89.6 on this")
print("      condition at N0; anything near zero here is the instrument.")

CONFIRM_SPEND = None            # <-- set to the number in the COST line

if spend_gate(n_calls, CONFIRM_SPEND, CTRL_A_OUT,
              factors=(("scenes", len(CALL_SCENES)), ("models", len(MODELS)),
                       ("repeats", CTRL_REPEATS))):
    S.run(str(CAPTURES), out_path=str(CTRL_A_OUT), models=MODELS,
          conditions=("congruent_face",), preferences=(PREFERENCE,),
          rungs=("N-CD",), modalities=("V",), kind="pair",
          repeats=CTRL_REPEATS)
    print("answered now:", answered(CTRL_A_OUT))

COST: 68 scenes x 2 models x 1 repeat = 136 calls
      congruent_face at N-CD. The face and the rule are both given,
      so only the conversion is at stake. GPT read 89.6 on this
      condition at N0; anything near zero here is the instrument.
already answered: 136 of 136 in ex2_q3_ctrlA_congruent_face_N-CD.jsonl
set CONFIRM_SPEND = 136 in this cell to proceed

not confirmed; no calls made.


In [6]:
# --- Control B. dims at N-CD. MAKES MODEL CALLS. ----------------------------
CTRL_B_OUT = RUNS / "ex2_q3_ctrlB_dims_N-CD.jsonl"
CTRL_REPEATS = 1

# ITS OWN FILE, not runs/ex2_q3_dims_N-CD.jsonl. The trial_id carries the
# rung and the repeat but not the date, so a fresh repeat 1 written into the
# old file would be skipped as already answered and this cell would report
# itself complete having made no calls. That is exactly the failure it
# exists to detect, and it would report a pass.
n_calls = len(CALL_SCENES) * len(MODELS) * CTRL_REPEATS
print("COST: %d scenes x %d models x %d repeat = %d calls"
      % (len(CALL_SCENES), len(MODELS), CTRL_REPEATS, n_calls))
print("      dims at N-CD, the baseline both step-1 treatments are read")
print("      from. Must reproduce GPT 32.3 [23.3, 41.3] and Claude 0.0.")

CONFIRM_SPEND = None            # <-- set to the number in the COST line

if spend_gate(n_calls, CONFIRM_SPEND, CTRL_B_OUT,
              factors=(("scenes", len(CALL_SCENES)), ("models", len(MODELS)),
                       ("repeats", CTRL_REPEATS))):
    S.run(str(CAPTURES), out_path=str(CTRL_B_OUT), models=MODELS,
          conditions=("dims",), preferences=(PREFERENCE,),
          rungs=("N-CD",), modalities=("V",), kind="pair",
          repeats=CTRL_REPEATS)
    print("answered now:", answered(CTRL_B_OUT))

COST: 68 scenes x 2 models x 1 repeat = 136 calls
      dims at N-CD, the baseline both step-1 treatments are read
      from. Must reproduce GPT 32.3 [23.3, 41.3] and Claude 0.0.
already answered: 136 of 136 in ex2_q3_ctrlB_dims_N-CD.jsonl
set CONFIRM_SPEND = 136 in this cell to proceed

not confirmed; no calls made.


## Read-out

No model calls. Reads every file off disk, so it survives a kernel restart and
can be run part-way through a paid cell.

It prints the controls first and **refuses to report the treatments until both
pass**. It also prints the recorded predictions from `prompts.PREDICTIONS`
before any number, so that what was expected is on the page above what
happened.

In [7]:
# --- Read-out. No model calls. ---------------------------------------------
FILES = {
    ("congruent_face", "N-CD",  "control A"): RUNS / "ex2_q3_ctrlA_congruent_face_N-CD.jsonl",
    ("dims",           "N-CD",  "control B"): RUNS / "ex2_q3_ctrlB_dims_N-CD.jsonl",
    ("dims",           "N-D",   "baseline"):  RUNS / "ex2_q3_dims_N-D.jsonl",
    ("dims",           "N-CD",  "baseline"):  RUNS / "ex2_q3_dims_N-CD.jsonl",
    ("dims",           "N-S",   "treatment"): RUNS / "ex2_q3_dims_N-S.jsonl",
    ("dims",           "N-BCD", "treatment"): RUNS / "ex2_q3_dims_N-BCD.jsonl",
    ("dims",           "N-ACD", "baseline"):  RUNS / "ex2_q3_dims_N-ACD.jsonl",
    ("dims",           "N-ABCS", "ceiling"):  RUNS / "ex2_q3_dims_N-ABCS.jsonl",
}
TOL = 0.006

def cells(path, cond):
    if not pathlib.Path(path).exists():
        return None
    rows, _ = load_run(path, cond, MODELS)
    return keep_analysable(rows, USABLE)

def summarise(rows, model):
    """Every quantity this design reads, for one cell of it.

    Conversion is measured ONLY on replies that named the face correctly.
    A model that reads the wrong face and then converts it faithfully has
    not failed step 2, and scoring it as though it had would move the two
    steps' numbers together and hide which one a treatment repaired. The
    denominator is therefore n_face_ok, which is reported beside it because
    it shrinks exactly where reading is worst and the rate is noisiest.
    """
    sub = [r for r in rows if r["model"] == model]
    if not sub:
        return None
    told = [r for r in sub if r.get("resting_face")]
    fok = [r for r in told if r["resting_face"] == r["face"]]
    conv = [r for r in fok if r.get("opening_needed_m") is not None
            and abs(r["opening_needed_m"] - FACTS[r["resting_face"]]["grasp_m"]) <= TOL]
    d = [x for _, x in paired_diffs(sub, USABLE, "small_face", "large_face")]
    mean, lo, hi, npos = paired_mean_ci(d)
    fk, fn = full_flip_count(d)
    flo, fhi = wilson(fk, fn)
    face_lo, face_hi = wilson(len(fok), len(told)) if told else (None, None)
    conv_lo, conv_hi = wilson(len(conv), len(fok)) if fok else (None, None)
    return dict(n=len(sub), n_told=len(told), n_face_ok=len(fok),
                face=pct(len(fok), len(told)) if told else None,
                face_lo=face_lo, face_hi=face_hi,
                conv=pct(len(conv), len(fok)) if fok else None,
                conv_lo=conv_lo, conv_hi=conv_hi,
                franka_small=share_at([r for r in sub
                                       if r["face"] == "small_face"]),
                franka_large=share_at([r for r in sub
                                       if r["face"] == "large_face"]),
                delta=mean, lo=lo, hi=hi, npos=npos,
                flips=fk, flips_n=fn, flips_lo=flo, flips_hi=fhi)

loaded = {k: cells(v, k[0]) for k, v in FILES.items()}

print("=" * 72)
print("RECORDED PREDICTIONS, from prompts.PREDICTIONS, committed before the")
print("first call of either treatment")
print("=" * 72)
for _f in ("staged", "description"):
    print("  %-12s %s" % (_f, P.PREDICTIONS[_f]))
print()

# --- the controls, and the gate --------------------------------------------
print("=" * 72)
print("CONTROLS")
print("=" * 72)
ctrl_rows, ctrl_ok = [], True
for key, label, test in (
        (("congruent_face", "N-CD", "control A"), "A  congruent_face N-CD",
         lambda s: s["delta"] is not None and s["delta"] >= 80.0),
        (("dims", "N-CD", "control B"), "B  dims N-CD",
         lambda s: s["delta"] is not None and s["delta"] == s["delta"])):
    rows = loaded[key]
    if rows is None:
        print("  %-24s NOT RUN" % label); ctrl_ok = False; continue
    for m in MODELS:
        s = summarise(rows, m)
        if s is None:
            print("  %-24s %-10s no rows" % (label, m)); ctrl_ok = False; continue
        ctrl_rows.append([label, m, s["n"], fmt(s["face"]), fmt(s["conv"]),
                          fmt(s["delta"]), fmt(s["lo"]), fmt(s["hi"])])
show(["control", "model", "n", "face%", "conv%", "delta", "lo", "hi"], ctrl_rows)
print()
print("  Control A passes if GPT's contrast is high: the face and the rule")
print("  are both given, and it read 89.6 on this condition at N0 without")
print("  the rule. Control B passes if GPT reproduces 32.3 [23.3, 41.3] and")
print("  Claude stays at zero. One repeat gives a wider interval than the")
print("  three-repeat runs these are compared with, so read overlap rather")
print("  than equality.")
print()

# --- the treatments, gated -------------------------------------------------
_have_treat = any(loaded[k] for k in loaded if k[2] == "treatment")
if not ctrl_ok:
    print("=" * 72)
    print("TREATMENTS NOT REPORTED: a control has not been run or returned no")
    print("rows. A treatment measured against a baseline the models no longer")
    print("reproduce is not a measurement. Run the control cells first.")
    print("=" * 72)
elif not _have_treat:
    print("Controls are in. Neither treatment has been run yet.")
else:
    print("=" * 72)
    print("TREATMENTS, each against the configuration it is read from")
    print("=" * 72)
    out = []
    for key, ref, step in (
            (("dims", "N-S", "treatment"), ("dims", "N-D", "baseline"),
             "2, conversion"),
            (("dims", "N-BCD", "treatment"), ("dims", "N-CD", "baseline"),
             "1, face reading")):
        rows, base = loaded[key], loaded[ref]
        for m in MODELS:
            s = summarise(rows, m) if rows else None
            b = summarise(base, m) if base else None
            f = lambda d, k: "--" if not d else fmt(d[k])
            out.append([key[1], m, step, f(b, "face"), f(s, "face"),
                        f(b, "conv"), f(s, "conv"),
                        f(b, "delta"), f(s, "delta")])
    show(["rung", "model", "step", "face% from", "face% to",
          "conv% from", "conv% to", "delta from", "delta to"], out)
    print()
    print("  N-S is read on the conversion column, N-BCD on the face column.")
    print("  The delta columns are carried for both because a treatment that")
    print("  moves its own step and not the allocation is a different finding")
    print("  from one that moves neither.")

    # --- the ceiling, reported apart because it attributes nothing ---------
    _ceil = loaded[("dims", "N-ABCS", "ceiling")]
    if _ceil:
        print()
        print("=" * 72)
        print("CEILING CELL, all four treatments at once")
        print("=" * 72)
        print("  prediction: %s" % P.CEILING_PREDICTION)
        print()
        crow = []
        for m in MODELS:
            c = summarise(_ceil, m)
            b = summarise(loaded[("dims", "N-D", "baseline")], m)
            if c is None:
                continue
            crow.append([m, c["n"], fmt(c["face"]), fmt(c["face_lo"]),
                         fmt(c["face_hi"]), fmt(c["conv"]), fmt(c["delta"]),
                         fmt(c["lo"]), fmt(c["hi"]),
                         "%d/%d" % (c["flips"], c["flips_n"]),
                         fmt(b["delta"]) if b else "--"])
        show(["model", "n", "face%", "face lo", "face hi", "conv%",
              "delta", "lo", "hi", "flips", "baseline delta"], crow)
        print()
        print("  Read against the baseline column only. This cell differs")
        print("  from every other by more than one change, so it gives the")
        print("  height reached and attributes none of it.")

# --- THE EXTRACT. One long-format row per cell, every quantity, always ------
# Written whatever has been run so far, with the missing cells absent rather
# than blank, so the file is a record of what exists rather than a shape that
# has to be filled in. Long format on purpose: the chapter's tables are not
# settled, and a wide file built for one of them has to be regenerated for
# the next, whereas any of them can be pivoted out of this.
COLS = ["condition", "rung", "role", "model", "n", "n_told", "n_face_ok",
        "face_pct", "face_lo", "face_hi", "conv_pct", "conv_lo", "conv_hi",
        "franka_small_pct", "franka_large_pct", "delta", "delta_lo",
        "delta_hi", "n_positions", "flips", "flips_of", "flips_lo",
        "flips_hi"]
long_rows = []
for (cond, rung, role), rows in sorted(loaded.items()):
    if rows is None:
        continue
    for m in MODELS:
        d = summarise(rows, m)
        if d is None:
            continue
        long_rows.append([cond, rung, role, m, d["n"], d["n_told"],
                          d["n_face_ok"], fmt(d["face"]), fmt(d["face_lo"]),
                          fmt(d["face_hi"]), fmt(d["conv"]), fmt(d["conv_lo"]),
                          fmt(d["conv_hi"]), fmt(d["franka_small"]),
                          fmt(d["franka_large"]), fmt(d["delta"]),
                          fmt(d["lo"]), fmt(d["hi"]), d["npos"], d["flips"],
                          d["flips_n"], fmt(d["flips_lo"]),
                          fmt(d["flips_hi"])])
print()
if long_rows:
    write_csv("tab_ex2_q3_repair_cells.csv", COLS, long_rows)
else:
    print("nothing run yet, so no extract written")

RECORDED PREDICTIONS, from prompts.PREDICTIONS, committed before the
first call of either treatment
  staged       moves conversion in dims, read from N-D: GPT converts a correctly named face on 65.1 percent of such replies there and the relation is predicted to be present but unapplied, so requiring the intermediate should raise it. Inert would mean the relation was genuinely absent
  description  moves face reading in dims, read from N-CD, where face accuracy is 66.1 percent. Inert would mean the model was not short of a set of poses to match the image against, which given attention's 10.5-point gain at N-ACD would locate the loss in consulting the image rather than in describing it

CONTROLS
control                 model      n   face%  conv%  delta  lo    hi   
----------------------  ---------  --  -----  -----  -----  ----  -----
A  congruent_face N-CD  gpt_hi     64  100.0  96.9   93.8   85.2  102.3
A  congruent_face N-CD  claude_md  64  100.0  78.1   56.2   38.8  73.7 
B  dims 

## Treatments

**Make model calls,** three repeats each, about \num{816} calls in total.

`N-S` is the staged report: the baseline instruction with one required field
between the face and the opening, and no relation stated. It is read against
`N-D`, already on disk.

`N-BCD` is the description: `candidate_faces_m` added to every object in the
state, and one sentence naming the field. It is read against `N-CD`, which
control B has just re-bought.

Do not run these until cell 5 says both controls passed.

In [8]:
# --- Treatment 1. dims at N-S. MAKES MODEL CALLS. ---------------------------
TREAT_S_OUT = RUNS / "ex2_q3_dims_N-S.jsonl"

n_calls = len(CALL_SCENES) * len(MODELS) * REPEATS
print("COST: %d scenes x %d models x %d repeats = %d calls"
      % (len(CALL_SCENES), len(MODELS), REPEATS, n_calls))
print("      dims at N-S, read against N-D. Conversion at N-D is 65.1%")
print("      for GPT on replies that named the face correctly.")

# The schema must carry the field the block asks for, checked here rather
# than discovered in the replies.
assert "horizontal_extents_m" in P.SCHEMAS[P.RUNGS["N-S"]["schema"]], (
    "N-S asks for an intermediate the answer schema gives it nowhere to put")

CONFIRM_SPEND = None            # <-- set to the number in the COST line

if spend_gate(n_calls, CONFIRM_SPEND, TREAT_S_OUT,
              factors=(("scenes", len(CALL_SCENES)), ("models", len(MODELS)),
                       ("repeats", REPEATS))):
    S.run(str(CAPTURES), out_path=str(TREAT_S_OUT), models=MODELS,
          conditions=("dims",), preferences=(PREFERENCE,),
          rungs=("N-S",), modalities=("V",), kind="pair", repeats=REPEATS)
    print("answered now:", answered(TREAT_S_OUT))

COST: 68 scenes x 2 models x 3 repeats = 408 calls
      dims at N-S, read against N-D. Conversion at N-D is 65.1%
      for GPT on replies that named the face correctly.
already answered: 408 of 408 in ex2_q3_dims_N-S.jsonl
set CONFIRM_SPEND = 408 in this cell to proceed

not confirmed; no calls made.


In [9]:
# --- Treatment 2. dims at N-BCD. MAKES MODEL CALLS. -------------------------
TREAT_B_OUT = RUNS / "ex2_q3_dims_N-BCD.jsonl"

n_calls = len(CALL_SCENES) * len(MODELS) * REPEATS
print("COST: %d scenes x %d models x %d repeats = %d calls"
      % (len(CALL_SCENES), len(MODELS), REPEATS, n_calls))
print("      dims at N-BCD, read against N-CD. Face accuracy at N-CD is")
print("      66.1% for GPT and 50.5% for Claude.")

# THE PROPERTY THE TREATMENT RESTS ON, re-checked at the point of spending.
# candidate_faces_m must be identical for the two poses at a position: if it
# is not, the state names the resting face and this cell measures obedience
# to a supplied answer. prompts asserts it over orderings; this asserts it
# over the scenes actually being bought.
_seen = {}
for _s in CALL_SCENES:
    _body, _meta = T.transform({"state": _s["state"],
                                "positions_exact": _s["positions_exact"]}, "dims")
    _pos = _s["seq"].rsplit("_", 1)[0]
    _seen.setdefault(_pos, {})[_meta["true_pose"]] = P.candidate_faces_signature(_body)
_leaky = [p for p, v in _seen.items() if len(v) == 2 and len(set(v.values())) != 1]
assert not _leaky, (
    "candidate_faces_m differs between the two poses at %s, so the state "
    "states the resting face and this cell measures nothing." % _leaky)
print("      candidate_faces_m checked: identical across both poses at all")
print("      %d positions, so it carries no pose information." % len(_seen))

CONFIRM_SPEND = None            # <-- set to the number in the COST line

if spend_gate(n_calls, CONFIRM_SPEND, TREAT_B_OUT,
              factors=(("scenes", len(CALL_SCENES)), ("models", len(MODELS)),
                       ("repeats", REPEATS))):
    S.run(str(CAPTURES), out_path=str(TREAT_B_OUT), models=MODELS,
          conditions=("dims",), preferences=(PREFERENCE,),
          rungs=("N-BCD",), modalities=("V",), kind="pair", repeats=REPEATS)
    print("answered now:", answered(TREAT_B_OUT))

COST: 68 scenes x 2 models x 3 repeats = 408 calls
      dims at N-BCD, read against N-CD. Face accuracy at N-CD is
      66.1% for GPT and 50.5% for Claude.
      candidate_faces_m checked: identical across both poses at all
      34 positions, so it carries no pose information.
already answered: 408 of 408 in ex2_q3_dims_N-BCD.jsonl
set CONFIRM_SPEND = 408 in this cell to proceed

not confirmed; no calls made.


## Ceiling cell

**Makes model calls,** three repeats, \num{408} calls.

`N-ABCS` applies all four treatments at once: attention, the description in
the state, the derivation sentence, and the staged report. It is not
diagnostic and is not meant to be. Attribution comes from the four single
cells; this one answers what the combination reaches, which is the question
the singles cannot.

It supports no increment. `N-ABCS` minus `N-ACD` is not the description's
contribution, because it also swaps the elicitation block for the staged one
and so carries two changes. The only contrast it can be read in is against the
baseline, which differs from it by all four treatments together.

Note that derivation already takes conversion to 100 per cent, so the staged
report cannot contribute through step 2 here and can act only through reading
or the arm.

**Run this last**, after the four cells above, so that the prediction recorded
in `prompts.CEILING_PREDICTION` is read against numbers that already exist.

In [10]:
# --- Ceiling cell. dims at N-ABCS. MAKES MODEL CALLS. -----------------------
CEIL_OUT = RUNS / "ex2_q3_dims_N-ABCS.jsonl"

print("THE PREDICTION, from prompts.CEILING_PREDICTION, recorded before this")
print("cell was called once:")
for _line in P.CEILING_PREDICTION.split(". "):
    print("   ", _line.strip().rstrip(".") + ".")
print()

n_calls = len(CALL_SCENES) * len(MODELS) * REPEATS
print("COST: %d scenes x %d models x %d repeats = %d calls"
      % (len(CALL_SCENES), len(MODELS), REPEATS, n_calls))
print("      dims at N-ABCS, all four treatments at once. Read against the")
print("      N-D baseline; it differs from every other cell by more than one")
print("      change and attributes nothing.")

# The same pose-invariance guard the N-BCD cell applies, because N-ABCS also
# carries the description and the field must state no pose here either.
_seen = {}
for _s in CALL_SCENES:
    _body, _meta = T.transform({"state": _s["state"],
                                "positions_exact": _s["positions_exact"]}, "dims")
    _pos = _s["seq"].rsplit("_", 1)[0]
    _seen.setdefault(_pos, {})[_meta["true_pose"]] = P.candidate_faces_signature(_body)
_leaky = [q for q, v in _seen.items() if len(v) == 2 and len(set(v.values())) != 1]
assert not _leaky, (
    "candidate_faces_m differs between the two poses at %s" % _leaky)
assert "N-ABCS" in P.CANDIDATE_FACE_RUNGS, (
    "N-ABCS carries the description block but is not in CANDIDATE_FACE_RUNGS, "
    "so the gloss would name a field the state does not carry")
print("      candidate_faces_m checked at all %d positions." % len(_seen))

CONFIRM_SPEND = None            # <-- set to the number in the COST line

if spend_gate(n_calls, CONFIRM_SPEND, CEIL_OUT,
              factors=(("scenes", len(CALL_SCENES)), ("models", len(MODELS)),
                       ("repeats", REPEATS))):
    S.run(str(CAPTURES), out_path=str(CEIL_OUT), models=MODELS,
          conditions=("dims",), preferences=(PREFERENCE,),
          rungs=("N-ABCS",), modalities=("V",), kind="pair", repeats=REPEATS)
    print("answered now:", answered(CEIL_OUT))

THE PREDICTION, from prompts.CEILING_PREDICTION, recorded before this
cell was called once:
    face accuracy above 76.6 percent and a contrast above 53.1, both short of 100, leaving a residual that no combination of supplying information and rephrasing the question closes.
    Claude stays at zero with face accuracy at chance, since nothing has moved its reading.
    A contrast reaching 100 would be a better result than the chapter currently claims is available and would have to be reported as such.

COST: 68 scenes x 2 models x 3 repeats = 408 calls
      dims at N-ABCS, all four treatments at once. Read against the
      N-D baseline; it differs from every other cell by more than one
      change and attributes nothing.
      candidate_faces_m checked at all 34 positions.
already answered: 0 of 408 in ex2_q3_dims_N-ABCS.jsonl
set CONFIRM_SPEND = 408 in this cell to proceed

not confirmed; no calls made.


## Provenance

No model calls. One row per input file, with its length, its SHA-256 and the
prompt version it was written under. This is the trail from a number in the
chapter back to the file it came from, and it is what lets a reader a month
later tell a file that was bought whole from one that was topped up.

In [11]:
# --- Provenance. No model calls. -------------------------------------------
today = datetime.date.today().isoformat()
prov = [provenance_row("captures", CAPTURES / "consults.jsonl", OUT,
                       today=today, default_version=P.EX2_PROMPT_VERSION)]
for (cond, rung, role), path in sorted(FILES.items()):
    prov.append(provenance_row("%s_%s_%s" % (role.replace(" ", ""), cond, rung),
                               path, OUT, today=today,
                               default_version=P.EX2_PROMPT_VERSION))

show(["role", "rows", "sha256", "prompt_version", "models"],
     [[r[0], r[2], (r[3] or "")[:12], r[4], r[5]] for r in prov])
write_csv("tab_ex2_q3_repair_provenance.csv",
          ["role", "path", "rows", "sha256", "prompt_version", "model_string",
           "run_date"], prov)
print()
print("A row reading MISSING is a cell that has not been bought. A row whose")
print("prompt_version is not %s was written under a different prompt and"
      % P.EX2_PROMPT_VERSION)
print("cannot be compared with the rest without saying so in the chapter.")

role                          rows  sha256        prompt_version  models                 
----------------------------  ----  ------------  --------------  -----------------------
captures                      102   b1579d5d8d1f  2026-08-27b                            
controlA_congruent_face_N-CD  141   58f2a5fdd0df  2026-08-27b     claude_md;gpt_hi       
ceiling_dims_N-ABCS           0     MISSING                                              
baseline_dims_N-ACD           612   4b56b8c44a59  2026-08-27b     claude_md;gemini;gpt_hi
treatment_dims_N-BCD          411   c2482246cac7  2026-08-27b     claude_md;gpt_hi       
baseline_dims_N-CD            612   aaa13ffc0bb1  2026-08-27b     claude_md;gemini;gpt_hi
controlB_dims_N-CD            136   0c6f0e997b99  2026-08-27b     claude_md;gpt_hi       
baseline_dims_N-D             614   67d9955d92db  2026-08-27b     claude_md;gemini;gpt_hi
treatment_dims_N-S            409   6b8a21924dfe  2026-08-27b     claude_md;gpt_hi       
wrote tabl

---
## Audit: Tables 5.12, 5.13 and 5.14 against the thesis

The three E2-C tables, each cell checked against the number Chapter 5 prints.

Table 5.14 is assembled from **two** notebooks. Its `N-CD` baseline and
`+ description` rows come from this one; its `+ attention` row is `N-ACD`,
bought and read in `ex2_q3_remediation.ipynb` and written to
`tables/ex2_q3/`. Both are checked here so the split cannot hide a drift in
half a table.

One rung wears two labels. `N-CD` is "+ derivation" in Table 5.13 and the
baseline in Table 5.14 — the same rows read for two different questions, which
is correct and easy to misread.

In [12]:
# --- Audit against the thesis. No model calls. ------------------------------
# PUBLISHED VALUES, transcribed from Chapter 5 and never computed here.

# Table 5.12, the two controls at one repeat: face %, conversion %, contrast.
THESIS_512 = {
    ("A", "congruent_face", "N-CD", "gpt_hi"):    (100.0, 96.9, 93.8),
    ("A", "congruent_face", "N-CD", "claude_md"): (100.0, 78.1, 56.2),
    ("B", "dims", "N-CD", "gpt_hi"):              (64.1, 97.6, 25.0),
    ("B", "dims", "N-CD", "claude_md"):           (43.8, 96.4, 0.0),
}

# Table 5.13, mapping a face to an opening: conversion, Franka share on each
# face, and the paired allocation contrast. All in dims.
THESIS_513 = {
    ("N-D", "gpt_hi"):     (65.1, 63.5, 60.4, 3.1),
    ("N-D", "claude_md"):  (68.0, 67.7, 67.7, 0.0),
    ("N-CD", "gpt_hi"):    (100.0, 100.0, 67.7, 32.3),
    ("N-CD", "claude_md"): (94.8, 100.0, 100.0, 0.0),
    ("N-S", "gpt_hi"):     (88.5, 88.5, 89.6, -1.0),
    ("N-S", "claude_md"):  (81.2, 82.3, 70.8, 11.5),
}

# Table 5.14, reading the face: face accuracy, both Franka shares, contrast
# and its paired interval. N-ACD comes from the remediation notebook.
THESIS_514 = {
    ("N-CD", "gpt_hi"):     (66.1, 100.0, 67.7, 32.3, 23.3, 41.3),
    ("N-CD", "claude_md"):  (50.5, 100.0, 100.0, 0.0, 0.0, 0.0),
    ("N-BCD", "gpt_hi"):    (75.0, 100.0, 50.0, 50.0, 39.8, 60.2),
    ("N-BCD", "claude_md"): (50.0, 40.6, 38.5, 2.1, -12.6, 16.7),
    ("N-ACD", "gpt_hi"):    (76.6, 100.0, 46.9, 53.1, 42.6, 63.6),
    ("N-ACD", "claude_md"): (50.0, 100.0, 100.0, 0.0, 0.0, 0.0),
}

EPS = 0.06
problems = []


def agrees(label, got, want, eps=EPS):
    if want is None:
        return
    if got is None or abs(got - want) > eps:
        problems.append("%s: computed %s, thesis prints %s" % (label, got, want))


def read_csv(path):
    with open(path) as fh:
        return list(csv.DictReader(fh))


def num(row, field):
    v = row.get(field, "")
    return float(v) if v not in ("", "NA", None) else None


cells = read_csv(TABLES / "tab_ex2_q3_repair_cells.csv")
by = {(r["condition"], r["rung"], r["model"], r["role"]): r for r in cells}


def pick(cond, rung, model, roles):
    for role in roles:
        if (cond, rung, model, role) in by:
            return by[(cond, rung, model, role)]
    return None


# --- Table 5.12 ------------------------------------------------------------
for (ctrl, cond, rung, model), (face, conv, d) in sorted(THESIS_512.items()):
    row = pick(cond, rung, model, ("control A", "control B"))
    tag = "5.12 control %s %s %s" % (ctrl, cond, model)
    if row is None:
        problems.append("%s: no row in tab_ex2_q3_repair_cells.csv" % tag)
        continue
    agrees(tag + " face", num(row, "face_pct"), face)
    agrees(tag + " conversion", num(row, "conv_pct"), conv)
    agrees(tag + " contrast", num(row, "delta"), d)

# --- Table 5.13 ------------------------------------------------------------
for (rung, model), (conv, fs, fl, d) in sorted(THESIS_513.items()):
    row = pick("dims", rung, model, ("baseline", "treatment"))
    tag = "5.13 %s %s" % (rung, model)
    if row is None:
        problems.append("%s: no row" % tag)
        continue
    agrees(tag + " conversion", num(row, "conv_pct"), conv)
    agrees(tag + " franka small", num(row, "franka_small_pct"), fs)
    agrees(tag + " franka large", num(row, "franka_large_pct"), fl)
    agrees(tag + " contrast", num(row, "delta"), d)

# --- Table 5.14, including the half this notebook does not produce ---------
# N-ACD lives in tables/ex2_q3/, written by the remediation notebook: its face
# accuracy in the ceiling table and its contrast in the ceiling-contrast one.
Q3 = TABLES.parent / "ex2_q3"
ceiling = {(r["model"], r["rung"]): r
           for r in read_csv(Q3 / "tab_ex2_q3_ceiling.csv")}
ceil_con = {(r["model"], r["rung"]): r
            for r in read_csv(Q3 / "tab_ex2_q3_ceiling_contrast.csv")}

for (rung, model), (face, fs, fl, d, lo, hi) in sorted(THESIS_514.items()):
    tag = "5.14 %s %s" % (rung, model)
    if rung == "N-ACD":
        c, cc = ceiling.get((model, rung)), ceil_con.get((model, rung))
        if not c or not cc:
            problems.append("%s: missing from the remediation notebook's tables" % tag)
            continue
        agrees(tag + " face", num(c, "face_correct_pct"), face)
        agrees(tag + " contrast", num(cc, "contrast_pts"), d)
        agrees(tag + " paired lo", num(cc, "contrast_lo"), lo)
        agrees(tag + " paired hi", num(cc, "contrast_hi"), hi)
        # The shares are not tabled anywhere; the contrast is their difference,
        # so checking both against it is the strongest available statement.
        agrees(tag + " shares agree with the contrast", round(fs - fl, 1), d)
        continue
    row = pick("dims", rung, model, ("baseline", "treatment"))
    if row is None:
        problems.append("%s: no row" % tag)
        continue
    agrees(tag + " face", num(row, "face_pct"), face)
    agrees(tag + " franka small", num(row, "franka_small_pct"), fs)
    agrees(tag + " franka large", num(row, "franka_large_pct"), fl)
    agrees(tag + " contrast", num(row, "delta"), d)
    agrees(tag + " paired lo", num(row, "delta_lo"), lo)
    agrees(tag + " paired hi", num(row, "delta_hi"), hi)

n_checks = len(THESIS_512) * 3 + len(THESIS_513) * 4 + len(THESIS_514) * 5
print("Experiment 2, Q3: tables checked against the thesis")
show(["Table", "Reports", "Source"],
     [["5.12", "the two controls", "tab_ex2_q3_repair_cells.csv"],
      ["5.13", "step 2, mapping a face to an opening", "tab_ex2_q3_repair_cells.csv"],
      ["5.14", "step 1, reading the face",
       "repair_cells.csv + ex2_q3/ceiling*.csv"]])
print()
for p in problems:
    print("  MISMATCH  %s" % p)
print("%d checks against Chapter 5, %d disagreed" % (n_checks, len(problems)))
assert not problems, "Q3 no longer reproduces the thesis: %s" % problems[:5]
print("every Q3 table still matches what Chapter 5 prints")

Experiment 2, Q3: tables checked against the thesis
Table  Reports                               Source                                
-----  ------------------------------------  --------------------------------------
5.12   the two controls                      tab_ex2_q3_repair_cells.csv           
5.13   step 2, mapping a face to an opening  tab_ex2_q3_repair_cells.csv           
5.14   step 1, reading the face              repair_cells.csv + ex2_q3/ceiling*.csv

66 checks against Chapter 5, 0 disagreed
every Q3 table still matches what Chapter 5 prints
